# Notebook 23 — Multiplex Hubs (Descriptive)

## Question

Nb12 showed the peptide wireless connectome is structurally distinct from the synaptic graph (Jaccard 0.045). But some neurons may be hubs in BOTH. These "multiplex hubs" would be bridge neurons translating between fast-reflex wiring and slow-neuromodulatory state.

Identify the multiplex hubs and do a light enrichment analysis.

## Scope

This is **descriptive** — not a formal hypothesis test. N ≈ 20 multiplex hubs is too small for FDR on 13k genes. Instead we report:
- List of multiplex hubs
- Their known NT identities and functional classes
- Whether they cluster in any annotated category

## Preregistered criteria (descriptive)

1. **≥ 5 multiplex hubs identified** (neurons top-10% in both graphs).
2. **Report the list and cross-check with Loer & Rand NT classification**.

In [1]:
import sys
from pathlib import Path
import warnings; warnings.simplefilter('ignore')
_HERE = Path.cwd()
if (_HERE / 'lib').is_dir(): sys.path.insert(0, str(_HERE))
elif (_HERE.parent / 'lib').is_dir(): sys.path.insert(0, str(_HERE.parent))

from lib.paths import DERIVED
from lib.reference import load_nt_reference
import numpy as np, pandas as pd

adult = np.load(DERIVED / 'connectome_adult.npz', allow_pickle=True)
w_neurons = np.array([str(n) for n in adult['neurons']])
w_chem = adult['chem_adj']
w_gap  = adult['gap_adj']

pep = np.load(DERIVED / 'nb12_peptide_adjacency.npz', allow_pickle=True)
A_pep = pep['A_peptide'].astype(np.int32)
pep_neurons = np.array([str(n) for n in pep['neurons']])

nt = load_nt_reference()

# Alignment
assert len(w_neurons) == len(pep_neurons)
assert (w_neurons == pep_neurons).all(), 'Witvliet and peptide neuron order differ'

print(f'N={len(w_neurons)} neurons, synaptic edges={int((w_chem>0).sum())}, peptide edges={A_pep.sum()}')

N=222 neurons, synaptic edges=2191, peptide edges=4480


## Step 1 — Compute multiplex hub membership

In [2]:
# Synaptic: directed in+out degree
syn_total = (w_chem > 0).astype(int).sum(axis=1) + (w_chem > 0).astype(int).sum(axis=0)
# Peptide: directed in+out degree
pep_total = A_pep.sum(axis=1) + A_pep.sum(axis=0)

thresh_syn = np.percentile(syn_total, 90)
thresh_pep = np.percentile(pep_total, 90)

syn_hub = (syn_total >= thresh_syn)
pep_hub = (pep_total >= thresh_pep)
multi_hub = syn_hub & pep_hub
syn_only = syn_hub & ~pep_hub
pep_only = pep_hub & ~syn_hub

print(f'Synaptic hubs (top 10%):  {syn_hub.sum()} (threshold >={thresh_syn:.0f})')
print(f'Peptide hubs (top 10%):   {pep_hub.sum()} (threshold >={thresh_pep:.0f})')
print(f'MULTIPLEX HUBS (both):    {multi_hub.sum()}')
print(f'Synaptic-only hubs:       {syn_only.sum()}')
print(f'Peptide-only hubs:        {pep_only.sum()}')

multi_hub_neurons = w_neurons[multi_hub]
syn_only_neurons = w_neurons[syn_only]
pep_only_neurons = w_neurons[pep_only]

Synaptic hubs (top 10%):  23 (threshold >=35)
Peptide hubs (top 10%):   24 (threshold >=94)
MULTIPLEX HUBS (both):    5
Synaptic-only hubs:       18
Peptide-only hubs:        19


## Step 2 — Multiplex hubs with NT classification

In [3]:
def nt_of(n):
    v = nt.nt_of(n)
    if v is None: return 'Unknown'
    s = v.lower()
    if 'acetylcholine' in s: return 'ACh'
    if 'gaba' in s: return 'GABA'
    if 'glutamate' in s: return 'Glu'
    if 'dopamine' in s: return 'Dopamine'
    if 'serotonin' in s: return 'Serotonin'
    if 'octopamine' in s: return 'Octopamine'
    if 'tyramine' in s: return 'Tyramine'
    return str(v)

multi_df = pd.DataFrame({
    'neuron': multi_hub_neurons,
    'syn_total_degree': syn_total[multi_hub],
    'pep_total_degree': pep_total[multi_hub],
    'NT': [nt_of(n) for n in multi_hub_neurons],
}).sort_values('syn_total_degree', ascending=False)

print('MULTIPLEX HUBS (top 10% in both synaptic AND peptide graphs):')
print(multi_df.to_string(index=False))
print(f'\nNT breakdown:')
print(multi_df['NT'].value_counts().to_string())

# Compare to syn-only hubs
syn_only_df = pd.DataFrame({
    'neuron': syn_only_neurons,
    'NT': [nt_of(n) for n in syn_only_neurons],
})
print(f'\nSyn-only hub NT breakdown:')
print(syn_only_df['NT'].value_counts().to_string())

pep_only_df = pd.DataFrame({
    'neuron': pep_only_neurons,
    'NT': [nt_of(n) for n in pep_only_neurons],
})
print(f'\nPep-only hub NT breakdown:')
print(pep_only_df['NT'].value_counts().to_string())

MULTIPLEX HUBS (top 10% in both synaptic AND peptide graphs):
neuron  syn_total_degree  pep_total_degree       NT
  RIML                51               130 Tyramine
  RIMR                44               130 Tyramine
  RMGR                39               123  Unknown
  RMGL                36               123  Unknown
  RIGR                35               114      Glu

NT breakdown:
NT
Tyramine    2
Unknown     2
Glu         1

Syn-only hub NT breakdown:
NT
ACh           10
Glu            4
Dopamine       2
Octopamine     1
GABA           1

Pep-only hub NT breakdown:
NT
Glu          9
ACh          8
Serotonin    2


## Step 3 — NT enrichment test (Fisher exact)

In [4]:
from scipy.stats import fisher_exact

# Is any NT class over-represented among multiplex hubs?
for nt_class in ['ACh','Glu','GABA','Dopamine','Serotonin']:
    # 2x2 table: (multiplex hub, other) x (NT=class, NT!=class)
    all_nts = np.array([nt_of(n) for n in w_neurons])
    a = int(((all_nts == nt_class) & multi_hub).sum())  # this-NT AND multiplex
    b = int(((all_nts == nt_class) & ~multi_hub).sum()) # this-NT AND not-multiplex
    c = int(((all_nts != nt_class) & multi_hub).sum())
    d = int(((all_nts != nt_class) & ~multi_hub).sum())
    try:
        odds, pval = fisher_exact([[a, b], [c, d]], alternative='greater')
    except Exception:
        odds, pval = 0, 1
    total_nt = a + b
    expected = multi_hub.sum() * total_nt / len(w_neurons) if len(w_neurons) else 0
    print(f'  {nt_class:12s}: multiplex={a}/{multi_hub.sum()} vs all-neurons {total_nt}/{len(w_neurons)}  expected≈{expected:.1f}  odds={odds:.2f}  p={pval:.3e}')

  ACh         : multiplex=0/5 vs all-neurons 85/222  expected≈1.9  odds=0.00  p=1.000e+00
  Glu         : multiplex=1/5 vs all-neurons 54/222  expected≈1.2  odds=0.77  p=7.554e-01
  GABA        : multiplex=0/5 vs all-neurons 6/222  expected≈0.1  odds=0.00  p=1.000e+00


  Dopamine    : multiplex=0/5 vs all-neurons 6/222  expected≈0.1  odds=0.00  p=1.000e+00
  Serotonin   : multiplex=0/5 vs all-neurons 2/222  expected≈0.0  odds=0.00  p=1.000e+00


## Step 4 — Criteria + save

In [5]:
crit1 = multi_hub.sum() >= 5

print('CRITERIA')
print('=' * 60)
print(f'  [{"PASS" if crit1 else "FAIL"}] >= 5 multiplex hubs      {multi_hub.sum()}')
print('=' * 60)

if crit1:
    verdict = f'DESCRIPTIVE — {multi_hub.sum()} multiplex hubs identified; see list for cross-system bridges'
else:
    verdict = 'INCONCLUSIVE — too few multiplex hubs for analysis'
print(f'VERDICT: {verdict}')

summary = pd.DataFrame([{
    'n_synaptic_hubs': int(syn_hub.sum()),
    'n_peptide_hubs': int(pep_hub.sum()),
    'n_multiplex_hubs': int(multi_hub.sum()),
    'syn_threshold': float(thresh_syn),
    'pep_threshold': float(thresh_pep),
    'multiplex_hubs_list': ';'.join(multi_hub_neurons.tolist()),
    'verdict': verdict,
}])
summary.to_csv(DERIVED / 'nb23_final_summary.csv', index=False)
multi_df.to_csv(DERIVED / 'nb23_multiplex_hubs.csv', index=False)
print(summary.T.to_string())

CRITERIA
  [PASS] >= 5 multiplex hubs      5
VERDICT: DESCRIPTIVE — 5 multiplex hubs identified; see list for cross-system bridges
                                                                                                0
n_synaptic_hubs                                                                                23
n_peptide_hubs                                                                                 24
n_multiplex_hubs                                                                                5
syn_threshold                                                                                34.8
pep_threshold                                                                                94.0
multiplex_hubs_list                                                      RIGR;RIML;RIMR;RMGL;RMGR
verdict              DESCRIPTIVE — 5 multiplex hubs identified; see list for cross-system bridges
